<a href="https://colab.research.google.com/github/Masiania-bit/Maria-palander/blob/main/notebooks/week3_visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Week 3: Visualization — Визуализация

📥 [0] Подготовка данных: клонирование репозитория и загрузка CSV о вулканах

**Что делаем:**

- Клонируем GitHub-репозиторий курса в Google Colab
- Загружаем CSV-файл `data/volcano.csv` (2388 записей о вулканах)
- Переименовываем технические столбцы с URL Wikidata (volcano, country, mountainRange) в `*_url` (сохраняем для отладки)
- Переименовываем читаемые столбцы с постфиксом `Label` (volcanoLabel → volcano, countryLabel → country, mountainRangeLabel → mountainRange)
- Приводим числовое поле `elevation` к целочисленному типу `int`, заменяя пропуски на 0

**Результат:** чистая таблица `df` с полями:

- `volcano_url` — ссылка на объект Wikidata (для отладки)
- `country_url` — ссылка на страну в Wikidata
- `mountainRange_url` — ссылка на горную цепь в Wikidata
- `volcano` — название вулкана
- `elevation` — высота вулкана (метры)
- `country` — страна (если указана)
- `mountainRange` — горная цепь (если указана)
- `coord` — координаты в формате Wikidata (требует дополнительной обработки)

In [8]:
# 📥 [0] Клонирование репозитория и загрузка данных о вулканах

import os
import pandas as pd
import numpy as np

# Шаг 1: Клонируем ваш репозиторий
github_user = "MAsiania-bit"
repo = "Maria-palander"

repo_path = f"/content/{repo}"
if not os.path.exists(repo_path):
    !git clone -q https://github.com/{github_user}/{repo}.git
if os.getcwd() != repo_path:
    %cd {repo_path}

print("✅ Репозиторий готов\n")

# Шаг 2: Загружаем CSV-файл с вулканами
df = pd.read_csv("data/volcano.csv")
print(f"📊 Загружено строк в датасете: {len(df)}\n")

# Шаг 3: Очистка данных
# Технические столбцы с URL переименовываем, добавляя суффикс "_url"
rename_dict = {}
if "volcano" in df.columns:
    rename_dict["volcano"] = "volcano_url"
if "country" in df.columns:
    rename_dict["country"] = "country_url"
if "mountainRange" in df.columns:
    rename_dict["mountainRange"] = "mountainRange_url"
df = df.rename(columns=rename_dict)

# Столбцы с читаемыми названиями (Label) переименовываем в основные имена
rename_dict_label = {}
if "volcanoLabel" in df.columns:
    rename_dict_label["volcanoLabel"] = "volcano"
if "countryLabel" in df.columns:
    rename_dict_label["countryLabel"] = "country"
if "mountainRangeLabel" in df.columns:
    rename_dict_label["mountainRangeLabel"] = "mountainRange"
df = df.rename(columns=rename_dict_label)

# Приводим высоту к целочисленному типу (заменяем некорректные значения на 0)
df["elevation"] = pd.to_numeric(df["elevation"], errors="coerce").fillna(0).astype(int)

print("✅ Данные очищены\n")

# Шаг 4: Краткий обзор датасета
print("📋 Структура данных о вулканах:")
print(f"   Столбцы: {', '.join(df.columns)}")
if "volcano" in df.columns:
    print(f"   Количество вулканов: {df['volcano'].nunique()}")
if "country" in df.columns:
    print(f"   Уникальных стран: {df['country'].nunique()}")
if "mountainRange" in df.columns:
    print(f"   Уникальных горных цепей: {df['mountainRange'].nunique()}")
print(f"   Диапазон высот: {df['elevation'].min()} — {df['elevation'].max()} м")
print(f"   Средняя высота: {df['elevation'].mean():.1f} м\n")

# Дополнительная информация о подводных вулканах и аномалиях
submarine = df[df["elevation"] < 0]
if len(submarine) > 0:
    print(f"🌊 Подводных вулканов (высота < 0): {len(submarine)}")
    print(f"   Самая глубокая точка: {submarine['elevation'].min()} м\n")

extreme = df[df["elevation"] > 8000]
if len(extreme) > 0:
    print(f"🌋 Вулканов выше 8000 м (возможно, марсианские): {len(extreme)}")
    print(extreme[["volcano", "elevation", "country", "mountainRange"]].head())

print(f"\n🔍 Первые 5 строк:\n{df.head()}")

/content/Maria-palander
✅ Репозиторий готов

📊 Загружено строк в датасете: 2388

✅ Данные очищены

📋 Структура данных о вулканах:
   Столбцы: volcano_url, volcano, elevation, country_url, country, coord, mountainRange_url, mountainRange
   Количество вулканов: 2134
   Уникальных стран: 86
   Уникальных горных цепей: 135
   Диапазон высот: -1400 — 18225 м
   Средняя высота: 1089.1 м

🌊 Подводных вулканов (высота < 0): 5
   Самая глубокая точка: -1400 м

🌋 Вулканов выше 8000 м (возможно, марсианские): 5
           volcano  elevation country        mountainRange
0  Гора Аскрийская      18225     NaN                  NaN
1     Mount Tehama       9239     США  California Cascades
2    Hayes Volcano       9147     США  Tordrillo Mountains
3  Cerro Nicholson       8282    Перу                  NaN
4  Isanotski Peaks       8106     США     Алеутский хребет

🔍 Первые 5 строк:
                                volcano_url          volcano  elevation  \
0    http://www.wikidata.org/entity/Q499164  

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## [1] Bar Chart: Top 10 Countries

**Что показывает:** Лидеры по производству мультфильмов (США ~50% рынка). **Когда использовать:** Сравнение категорий по одному показателю. **Библиотеки:** `matplotlib`. **Особенность:** Логарифмическая шкала для больших разбросов.